In [1]:
import json
import re
import string
from collections import Counter

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    if not isinstance(s, str):
        s = str(s)
        
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    """Calculates if the normalized strings match exactly."""
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score(prediction, ground_truth):
    """Calculates the token-level F1 score (partial credit)."""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    
    # Count common tokens
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def evaluate_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    em_total = 0
    f1_total = 0
    valid_records = 0

    for item in data:
        if 'answer' in item and 'prediction' in item:
            pred = item['prediction']
            ans = item['answer']
            
            em_total += exact_match_score(pred, ans)
            f1_total += f1_score(pred, ans)
            valid_records += 1

    em_percentage = (em_total / valid_records) * 100
    f1_percentage = (f1_total / valid_records) * 100
    
    print(f"Total Records Evaluated: {valid_records}")
    print(f"Exact Match (EM): {em_percentage:.2f}%")
    print(f"F1 Score: {f1_percentage:.2f}%")
    
    return em_percentage, f1_percentage

In [3]:
evaluate_dataset('results/predictions_qwen3.5:2b_nothink.json')

Total Records Evaluated: 200
Exact Match (EM): 33.50%
F1 Score: 41.02%


(33.5, 41.01525974025974)